# Machine Learning-Based Buyer Segmentation and Investment Profiling for Real Estate Market Intelligence

**Dataset note:** Synthetic practice data, not actual Parcl customer data. Findings are illustrative only.

This notebook contains executed outputs from the analysis run dated 13 November 2026. Run cells from top to bottom to reproduce.

In [1]:
import pandas as pd, numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

df = pd.read_csv('real_estate_buyer_synthetic_dataset.csv')
print('Raw dataset shape:', df.shape)
display(df.head())

Raw dataset shape: (604, 10)


In [2]:
print('Missing values before cleaning:')
display(df.isna().sum())
print('Exact duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().drop_duplicates(subset=['client_id'], keep='first').copy()
print('After duplicate removal:', df.shape)
df = df.dropna(subset=['region']).copy()
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')
df['age'] = ((pd.Timestamp('2026-11-11') - df['date_of_birth']).dt.days / 365.25).round()
df.loc[~df['age'].between(18,100), 'age'] = np.nan
df['satisfaction_score'] = pd.to_numeric(df['satisfaction_score'], errors='coerce')
print('Final dashboard-aligned shape:', df.shape)
assert len(df) == 594 and df['client_id'].nunique() == 594

Missing values before cleaning:


client_id              0
client_type            0
gender                 6
country                0
region                 6
date_of_birth          0
acquisition_purpose    0
loan_applied            0
referral_channel        6
satisfaction_score      6
dtype: int64

Exact duplicate rows: 4
After duplicate removal: (600, 10)
Final dashboard-aligned shape: (594, 11)


In [3]:
cat = ['client_type','gender','country','region','acquisition_purpose','loan_applied','referral_channel']
num = ['age','satisfaction_score']
prep = ColumnTransformer([
 ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num),
 ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), cat)
])
X = prep.fit_transform(df[cat + num])
display(df.groupby('acquisition_purpose').agg(buyers=('client_id','count'), satisfaction=('satisfaction_score','mean')))

                     buyers  satisfaction
acquisition_purpose                      
Investment              330      3.720365
Personal use            264      3.872587

In [4]:
rows=[]
for k in range(2,8):
 model=KMeans(n_clusters=k, random_state=42, n_init=10)
 labels=model.fit_predict(X)
 rows.append({'k':k,'inertia':model.inertia_,'silhouette':silhouette_score(X,labels)})
evaluation=pd.DataFrame(rows)
display(evaluation)
best_k=int(evaluation.loc[evaluation.silhouette.idxmax(),'k'])
df['kmeans_cluster']=KMeans(n_clusters=best_k,random_state=42,n_init=10).fit_predict(X)
df['hierarchical_cluster']=AgglomerativeClustering(n_clusters=best_k).fit_predict(X.toarray() if hasattr(X,'toarray') else X)
display(df.groupby('kmeans_cluster').agg(buyers=('client_id','count'), median_age=('age','median'), investment_share=('acquisition_purpose',lambda s:(s=='Investment').mean()), loan_share=('loan_applied',lambda s:(s=='Yes').mean()), satisfaction=('satisfaction_score','mean')))

   k      inertia  silhouette
0  2  3262.122057    0.112616
1  3  2987.619541    0.098121
2  4  2859.617285    0.085430
3  5  2734.855558    0.082104
4  6  2651.551135    0.078131
5  7  2590.809382    0.077894

                buyers  median_age  investment_share  loan_share  satisfaction
kmeans_cluster                                                                
0                  208        40.5          0.596154    0.485577      2.788462
1                  386        40.0          0.533679    0.497409      4.334211

## Interpretation and limitations
The selected K-Means solution is k=2 with silhouette 0.112616. This indicates weak cluster separation. Treat groups as exploratory profiles, not strongly distinct natural market segments. The source dataset is synthetic practice data and is not actual Parcl customer data.